# Setup & Daten Laden

In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Memory Growth für TF
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus: tf.config.experimental.set_memory_growth(gpu, True)

from data import load_data, prepare_splits, NUM_CLASSES, LABEL_DICT
from models import VARIANT_V2_MINIMAL_WITH_BN_GAP, build_model

DATA_PATH = "./../data/Galaxy10_DECals.h5"
BATCH_SIZE = 32
TEMPERATURE = 5.0
ALPHA = 0.1 

print("Lade Bilder und erstelle identische Splits...")
images_raw, labels = load_data(DATA_PATH)
split_folds, X_test, y_test = prepare_splits(
    labels, use_kfold=False, val_size=0.2, test_size=0.2, random_state=42
)
train_idx, val_idx = split_folds[0]

print("Lade Teacher-Logits von Festplatte...")
train_logits = np.load("teacher_train_logits.npy")
val_logits = np.load("teacher_val_logits.npy")

assert len(train_idx) == len(train_logits), "Fehler: Train-Logits passen nicht zum Daten-Split!"
assert len(val_idx) == len(val_logits), "Fehler: Val-Logits passen nicht zum Daten-Split!"

def prepare_tf_dataset(indices, teacher_logits, batch_size, is_training=True):
    x_data = images_raw[indices]
    y_data = labels[indices]
    dataset = tf.data.Dataset.from_tensor_slices((x_data, y_data, teacher_logits))
    if is_training:
        dataset = dataset.shuffle(buffer_size=1024)
    return dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)

train_dataset = prepare_tf_dataset(train_idx, train_logits, BATCH_SIZE, is_training=True)
val_dataset = prepare_tf_dataset(val_idx, val_logits, BATCH_SIZE, is_training=False)

# Distiller Klasse & Training

In [ ]:
class Distiller(keras.Model):
    def __init__(self, student, temperature=3.0, alpha=0.1):
        super().__init__()
        self.student = student
        self.temperature = temperature
        self.alpha = alpha
        self.student_loss_fn = keras.losses.SparseCategoricalCrossentropy(from_logits=False)
        self.distillation_loss_fn = keras.losses.KLDivergence()

    def compile(self, optimizer, metrics):
        super().compile(optimizer=optimizer, metrics=metrics)

    def train_step(self, data):
        x, y, teacher_logits = data
        with tf.GradientTape() as tape:
            student_preds = self.student(x, training=True)
            student_loss = self.student_loss_fn(y, student_preds)

            soft_teacher_probs = tf.nn.softmax(teacher_logits / self.temperature, axis=1)
            # Student gibt bereits Wahrscheinlichkeiten aus, wir tricksen für den Temperatur-Scale
            student_logits = tf.math.log(student_preds + 1e-7) 
            soft_student_probs = tf.nn.softmax(student_logits / self.temperature, axis=1)

            distillation_loss = self.distillation_loss_fn(soft_teacher_probs, soft_student_probs)
            loss = self.alpha * student_loss + (1 - self.alpha) * distillation_loss

        gradients = tape.gradient(loss, self.student.trainable_variables)
        self.optimizer.apply_gradients(zip(gradients, self.student.trainable_variables))
        self.compiled_metrics.update_state(y, student_preds)
        results = {m.name: m.result() for m in self.metrics}
        results.update({"stud_loss": student_loss, "dist_loss": distillation_loss})
        return results
        
    def test_step(self, data):
        x, y, _ = data
        y_prediction = self.student(x, training=False)
        student_loss = self.student_loss_fn(y, y_prediction)
        self.compiled_metrics.update_state(y, y_prediction)
        results = {m.name: m.result() for m in self.metrics}
        results.update({"val_loss": student_loss})
        return results
        
    def call(self, x):
        return self.student(x)

student_base = build_model(VARIANT_V2_MINIMAL_WITH_BN_GAP, learning_rate=1e-4)
student_base.summary()

distiller = Distiller(student=student_base, temperature=TEMPERATURE, alpha=ALPHA)
distiller.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    metrics=[keras.metrics.SparseCategoricalAccuracy(name="accuracy")]
)

print("Starte Knowledge Distillation Training...")
history = distiller.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=50,
    callbacks=[keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)]
)


# Evaluation & Plots

In [ ]:
X_test_tf = tf.data.Dataset.from_tensor_slices(images_raw[X_test]).batch(BATCH_SIZE)
y_pred_probs = student_base.predict(X_test_tf)
y_pred = np.argmax(y_pred_probs, axis=1)

test_acc = np.mean(y_pred == y_test)
print(f"\nTest Accuracy des kleinen Student-Modells: {test_acc * 100:.2f}%")

student_params = student_base.count_params() / 1_000_000 

models_data = {
    "Station 1 (V7)": {"params": 2.49, "acc": 82.36},
    "ResNet-50": {"params": 24.89, "acc": 38.64},
    "ResNet-152": {"params": 59.68, "acc": 43.55},
    "Inception V3": {"params": 22.54, "acc": 61.22},
    "Zoobot (Teacher)": {"params": 15.11, "acc": 89.12},
    "Dein Student (KD)": {"params": student_params, "acc": test_acc * 100}
}

fig, ax = plt.subplots(figsize=(12, 7))
for name, info in models_data.items():
    color = "red" if name == "Dein Student (KD)" else ("green" if "Zoobot" in name else "blue")
    marker = "*" if name == "Dein Student (KD)" else ("^" if "Zoobot" in name else "o")
    size = 300 if name == "Dein Student (KD)" else 150
    ax.scatter(info["params"], info["acc"], label=name, color=color, marker=marker, s=size)
    ax.text(info["params"] * 1.1, info["acc"], name, fontsize=10)

ax.set_xscale("log")
ax.set_xlabel("Parameteranzahl (Millionen, log-Skala)")
ax.set_ylabel("Test Accuracy (%)")
ax.set_title("Vergleich: Modellgröße vs. Performance (Ergebnisse Station 1, 2 & 3)")
ax.grid(True, which="both", ls="--", alpha=0.5)
plt.show()

cm = confusion_matrix(y_test, y_pred)
class_names = [LABEL_DICT[i] for i in range(NUM_CLASSES)]

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.title(f"Confusion Matrix - KD Student (Test Acc: {test_acc*100:.2f}%)")
plt.ylabel("Ground Truth")
plt.xlabel("Prediction")
plt.xticks(rotation=45, ha="right")
plt.show()